# Baseline Experiments

Systematic evaluation of baseline approaches for jaguar re-identification:
- **Random Baseline**: Random embeddings (sanity check, should get ~0% mAP)
- **Backbone-Only Baselines**: Pre-trained backbone features without fine-tuning (11 backbones)

**Tested Backbones:**
- **DINOv3-Large** (vit_large_patch16_dinov3.lvd1689m, 1024-dim, latest self-supervised ViT)
- **DINOv3-Base** (vit_base_patch16_dinov3.lvd1689m, 768-dim, efficient high-quality)
- **MegaDescriptor-L-384** (1536-dim, our main model, trained on wildlife datasets)
- **MegaDescriptor-B-224** (768-dim, animal re-ID specialist)
- DINOv2-Large (1024-dim, previous-generation self-supervised)
- DINOv2-Base (768-dim)
- DINOv2-Small (384-dim)
- ResNet50 (2048-dim)
- ConvNeXt Base (1024-dim)
- ConvNeXtV2 Base (1024-dim)
- EfficientNet B3 (1536-dim)

**Note**: Embeddings computed on-the-fly for all models.

Results logged to Wandb project: `camera-trap-reidentification`, group: `baselines`

## Setup

In [1]:
import sys
from pathlib import Path
import logging
import importlib

# Add src to path
project_root = Path.cwd().parent / "camera-trap-footage"
sys.path.insert(0, str(project_root / "src"))

from jaguars.common.logging_utils import setup_logger
from jaguars.reidentification.config import get_default_config
from jaguars.reidentification.experiments import get_baseline_experiments
from jaguars.reidentification.evaluation.evaluation import run_processing as run_evaluation

# Force reload to get the latest code
import jaguars.reidentification.experiments
importlib.reload(jaguars.reidentification.experiments)
from jaguars.reidentification.experiments import get_baseline_experiments

logger = setup_logger("baseline_experiments", level=logging.INFO)
print("✓ Imports successful")

✓ Imports successful


## Configuration

Configure base settings for all baseline experiments.

In [2]:
# Get default configuration
config = get_default_config()

# Wandb settings
config.wandb.enabled = True
config.wandb.entity = "jaguars"
config.wandb.project = "camera-trap-reidentification"
config.wandb.tags = ["baselines"]

# Dataset settings
config.dataset.source = "fiftyone"
config.dataset.fo_dataset_name = "JID_Master_Dataset"
config.dataset.fo_split_field = "closed_set_split"
config.dataset.fo_patches_field = "sam3_segmentations"
config.dataset.fo_label_field = "ground_truth"

# Training settings
config.training.num_epochs = 0  # Baselines don't train

print(f"✓ Base config loaded")
print(f"  Wandb project: {config.wandb.project}")
print(f"  Dataset: {config.dataset.fo_dataset_name}")

✓ Base config loaded
  Wandb project: camera-trap-reidentification
  Dataset: JID_Master_Dataset


## Get Baseline Experiments

In [3]:
# Get baseline experiments with our custom config
baseline_experiments = get_baseline_experiments(base_config=config)

print(f"✓ {len(baseline_experiments)} baseline experiments configured:")
for exp in baseline_experiments:
    print(f"  - {exp.name}: {exp.description}")

✓ 12 baseline experiments configured:
  - baseline_random: Random embeddings (sanity check - should be near 0% mAP)
  - baseline_hf-hub:BVRA_MegaDescriptor-L-384: BVRA MegaDescriptor Large 384 without fine-tuning
  - baseline_hf-hub:BVRA_MegaDescriptor-B-224: BVRA MegaDescriptor Base 224 without fine-tuning
  - baseline_vit_large_patch14_dinov2.lvd142m: DINOv2 Large without fine-tuning
  - baseline_vit_base_patch14_dinov2.lvd142m: DINOv2 Base without fine-tuning
  - baseline_vit_small_patch14_dinov2.lvd142m: DINOv2 Small without fine-tuning
  - baseline_resnet50: ResNet50 without fine-tuning
  - baseline_convnext_base: ConvNeXt Base without fine-tuning
  - baseline_convnextv2_base.fcmae_ft_in22k_in1k: ConvNeXtV2 Base without fine-tuning
  - baseline_efficientnet_b3: EfficientNet B3 without fine-tuning
  - baseline_vit_large_patch16_dinov3.lvd1689m: DINOv3 Large (1024-dim, latest self-supervised) without fine-tuning
  - baseline_vit_base_patch16_dinov3.lvd1689m: DINOv3 Base (768-dim, ef

## Run Experiments

Execute all baseline experiments and log to Wandb.

In [4]:
# Run all baseline experiments
results = {}

for experiment in baseline_experiments:
    logger.info(f"Running experiment: {experiment.name}")
    logger.info(f"  Description: {experiment.description}")
    logger.info(f"  Group: {experiment.group}")
    
    try:
        # Determine baseline mode based on loss name
        baseline_mode = experiment.base_config.baseline_mode
        
        # Run evaluation (for baselines, this skips model loading)
        result = run_evaluation(
            config=experiment.base_config,
            baseline_mode=baseline_mode,
            verbose=True
        )
        results[experiment.name] = result
        logger.info(f"✓ {experiment.name} completed")
    except Exception as e:
        logger.error(f"✗ {experiment.name} failed: {e}")
        import traceback
        traceback.print_exc()
        results[experiment.name] = {"error": str(e)}

print(f"\n✓ All {len(baseline_experiments)} baseline experiments completed")

03:37:20 - jid_logger.baseline_experiments - INFO - Running experiment: baseline_random
03:37:20 - jid_logger.baseline_experiments - INFO -   Description: Random embeddings (sanity check - should be near 0% mAP)
03:37:20 - jid_logger.baseline_experiments - INFO -   Group: baselines
03:37:20 - jid_logger.reidentification.evaluation - INFO - Starting evaluation...
03:37:20 - jid_logger.reidentification.evaluation - INFO - Dataset source: fiftyone
03:37:22 - jid_logger.reidentification.evaluation - INFO - Resource validation passed


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /sc/home/philipp.kolbe/.netrc.
wandb: Currently logged in as: hpi-philipp-kolbe to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


03:37:24 - jid_logger.reidentification.evaluation - INFO - Loading dataset...
03:37:36 - jid_logger.reidentification.evaluation - INFO - Train set: 946 samples, 76 classes
03:37:36 - jid_logger.reidentification.evaluation - INFO - Test set: 105 samples, 31 classes
03:37:36 - jid_logger.reidentification.evaluation - INFO - Running random baseline evaluation (no training)
03:37:36 - jid_logger.reidentification.evaluation - INFO - Computing comprehensive metrics...
03:37:36 - jid_logger.reidentification.evaluation - INFO - Sample-level mAP: 0.0831
03:37:36 - jid_logger.reidentification.evaluation - INFO - Identity-balanced mAP: 0.0602
03:37:36 - jid_logger.reidentification.evaluation - INFO - Closed-set mAP (≥3 train & test): 0.0855 (13/22 identities)
03:37:36 - jid_logger.reidentification.evaluation - INFO - CMC@1: 0.0312
03:37:36 - jid_logger.reidentification.evaluation - INFO - CMC@5: 0.1354
03:37:36 - jid_logger.reidentification.evaluation - INFO - Saved embeddings to data/results/rei

closed_set_map,▁
closed_set_num_identities,▁
cmc@1,▁
cmc@10,▁
cmc@20,▁
cmc@5,▁
identity_balanced_map,▁
map,▁
map_train_0-2_val_0-2,▁
map_train_1-2,▁
+13,...


03:37:39 - jid_logger.reidentification.evaluation - INFO - Evaluation completed!
03:37:39 - jid_logger.baseline_experiments - INFO - ✓ baseline_random completed
03:37:39 - jid_logger.baseline_experiments - INFO - Running experiment: baseline_hf-hub:BVRA_MegaDescriptor-L-384
03:37:39 - jid_logger.baseline_experiments - INFO -   Description: BVRA MegaDescriptor Large 384 without fine-tuning
03:37:39 - jid_logger.baseline_experiments - INFO -   Group: baselines
03:37:39 - jid_logger.reidentification.evaluation - INFO - Starting evaluation...
03:37:39 - jid_logger.reidentification.evaluation - INFO - Dataset source: fiftyone
03:37:39 - jid_logger.reidentification.evaluation - INFO - Resource validation passed


...

03:49:26 - jid_logger.reidentification.evaluation - INFO - Loading dataset...
03:49:38 - jid_logger.reidentification.evaluation - INFO - Train set: 946 samples, 76 classes
03:49:38 - jid_logger.reidentification.evaluation - INFO - Test set: 105 samples, 31 classes
03:49:38 - jid_logger.reidentification.evaluation - INFO - Running backbone-only evaluation (no fine-tuning)
Loading vit_base_patch16_dinov3.lvd1689m model...
Model loaded successfully
  Parameters: 85,641,216
  Embedding dimension: 768
03:49:41 - jid_logger.reidentification.evaluation - INFO - Extracting backbone embeddings...


Train embeddings: 100%|██████████| 30/30 [00:47<00:00,  1.59s/it]

03:50:33 - jid_logger.reidentification.evaluation - INFO - Computing comprehensive metrics...
03:50:33 - jid_logger.reidentification.evaluation - INFO - Sample-level mAP: 0.4688
03:50:33 - jid_logger.reidentification.evaluation - INFO - Identity-balanced mAP: 0.4838
03:50:33 - jid_logger.reidentification.evaluation - INFO - Closed-set mAP (≥3 train & test): 0.4375 (13/22 identities)
03:50:33 - jid_logger.reidentification.evaluation - INFO - CMC@1: 0.6458
03:50:33 - jid_logger.reidentification.evaluation - INFO - CMC@5: 0.7917
03:50:33 - jid_logger.reidentification.evaluation - INFO - Saved embeddings to data/results/reidentification/test_embeddings.npy
03:50:33 - jid_logger.reidentification.evaluation - INFO - Saved predictions to data/results/reidentification/test_predictions.json


closed_set_map,▁
closed_set_num_identities,▁
cmc@1,▁
cmc@10,▁
cmc@20,▁
cmc@5,▁
identity_balanced_map,▁
map,▁
map_train_0-2_val_0-2,▁
map_train_1-2,▁
+13,...


03:50:35 - jid_logger.reidentification.evaluation - INFO - Evaluation completed!
03:50:35 - jid_logger.baseline_experiments - INFO - ✓ baseline_vit_base_patch16_dinov3.lvd1689m completed

✓ All 12 baseline experiments completed


## Summary

Display results from all baseline experiments.

In [5]:
results

{'baseline_random': {'status': 'completed',
  'num_test_samples': 105,
  'num_classes': 31,
  'map': 0.08310865718583046,
  'identity_balanced_map': 0.06024284406903062,
  'cmc_curve': [0.03125,
   0.0625,
   0.07291666666666667,
   0.08333333333333333,
   0.13541666666666666,
   0.17708333333333334,
   0.20833333333333334,
   0.22916666666666666,
   0.23958333333333334,
   0.28125,
   0.3229166666666667,
   0.3958333333333333,
   0.4479166666666667,
   0.4791666666666667,
   0.4791666666666667,
   0.5416666666666666,
   0.5729166666666666,
   0.59375,
   0.6145833333333334,
   0.6458333333333334,
   0.6770833333333334,
   0.6770833333333334,
   0.6875,
   0.6875,
   0.6875,
   0.6875,
   0.6979166666666666,
   0.71875,
   0.7291666666666666,
   0.7291666666666666,
   0.7395833333333334,
   0.7604166666666666,
   0.7604166666666666,
   0.7708333333333334,
   0.7708333333333334,
   0.7708333333333334,
   0.7708333333333334,
   0.7708333333333334,
   0.7708333333333334,
   0.770833333333

In [6]:
# Print summary of results
import pandas as pd

summary_data = []
for exp_name, result in results.items():
    if "error" in result:
        summary_data.append({
            "Experiment": exp_name,
            "mAP": "ERROR",
            "CMC@1": "ERROR",
            "Closed-set mAP": "ERROR",
        })
    else:
        # Extract key metrics
        map_val = result.get("identity_balanced_map", "N/A")
        cmc1 = result.get("cmc@1", "N/A")
        cs_map = result.get("closed_set_map", "N/A")
        
        map_str = f"{map_val:.4f}" if isinstance(map_val, (int, float)) else str(map_val)
        cmc1_str = f"{cmc1:.4f}" if isinstance(cmc1, (int, float)) else str(cmc1)
        cs_str = f"{cs_map:.4f}" if isinstance(cs_map, (int, float)) else str(cs_map)
        
        summary_data.append({
            "Experiment": exp_name,
            "mAP": map_str,
            "CMC@1": cmc1_str,
            "Closed-set mAP": cs_str,
        })

summary_df = pd.DataFrame(summary_data)
print("\n=== Baseline Results Summary ===")
print(summary_df.to_string(index=False))
print(f"\nView detailed results at: https://wandb.ai/{config.wandb.entity}/{config.wandb.project}")


=== Baseline Results Summary ===
                                  Experiment    mAP  CMC@1 Closed-set mAP
                             baseline_random 0.0602 0.0312         0.0855
   baseline_hf-hub:BVRA_MegaDescriptor-L-384 0.4234 0.5938         0.3317
   baseline_hf-hub:BVRA_MegaDescriptor-B-224 0.3602 0.5729         0.3507
   baseline_vit_large_patch14_dinov2.lvd142m 0.3485 0.5521         0.3215
    baseline_vit_base_patch14_dinov2.lvd142m 0.3560 0.5208         0.3495
   baseline_vit_small_patch14_dinov2.lvd142m 0.4094 0.5521         0.3521
                           baseline_resnet50 0.3847 0.5521         0.3552
                      baseline_convnext_base 0.3803 0.5208         0.3369
baseline_convnextv2_base.fcmae_ft_in22k_in1k 0.3697 0.5625         0.3503
                    baseline_efficientnet_b3 0.3606 0.5208         0.3470
  baseline_vit_large_patch16_dinov3.lvd1689m 0.4427 0.6146         0.4231
   baseline_vit_base_patch16_dinov3.lvd1689m 0.4838 0.6458         0.4375

Vie